# Критерии оценки ДЗ 

> **Версия LanceDB: `lancedb<0.20` !!!**

Делаете прогоны на test_dataset, выбиваете максимум по метрикам. Я буду проверять на eval_dataset

---

## Базовые требования (4 балла)

| # | Критерий | Баллы |
|---|----------|-------|
| 1 | Описан процесс подготовки данных (`data_preporation.py`, `embedder.py`) | 1 |
| 2 | Описаны методы/функции для работы с векторной БД в `lance_db.py` (обязательно `read_all`) | 1 |
| 3 | Данные записаны в БД и можно выполнить `read_all` | 1 |
| 4 | Описан флоу retrieve и agent (`agent.py`, `retriever.py`) | 1 |

---

## Метрики качества (до 9 баллов)

### F1@K

| Порог    | Баллы |
|----------|-------|
| > 0.50   | 1     |
| > 0.75   | 2     |
| > 0.90   | 3     |

### NDCG@K

| Порог    | Баллы |
|----------|-------|
| > 0.65   | 1     |
| > 0.75   | 2     |
| > 0.90   | 3     |

### Faithfulness

| Порог    | Баллы |
|----------|-------|
| > 0.75   | 1     |
| > 0.85   | 2     |
| > 0.95   | 3     |

---

## Лог эволюции (до 2 баллов)

Оценивается индивидуально на основании полноты информации в `EVOLUTION.md`.

---

## Штрафы

| Условие | Штраф |
|---------|-------|
| Суммарный объём всех передаваемых для генерации чанков > 6000 символов | −100500 баллов |
| Использовали дополнительные библиотеки и не положили requirements.txt | −100500 баллов |

---

**Максимум: 15 баллов** (4 базовых + 9 метрики + 2 эволюция)

# Валидация RAG-агента через модуль `evaluation.py`

Демонстрация полного пайплайна оценки:
1. Подключаемся к векторному хранилищу и создаём агента
2. Загружаем тестовый датасет
3. Прогоняем агента на всех вопросах (многопоточно)
4. Вызываем `evaluate()` — получаем DataFrame с 7 метриками
5. Анализируем результаты

---
## 0. Подготовка

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# from src.config import Settings
from openai import OpenAI

In [2]:
client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key="pza_4U5FlDhFI2Z21-qHKB1K7pJBWmbwNheJ",
)

# MODEL = "deepseek/deepseek-v4-flash"
MODEL = "deepseek/deepseek-v4.1-flash"

---
## 1. Подключаемся к векторному хранилищу и создаём агента

In [3]:
import lancedb
# VECTORESTORE_DIR = Path(__file__).parent / "lance_db" / "vectorestore"
VECTORSTORE_DIR = "lance_db/vectorestore"
TABLE_NAME = "chunks"

db = lancedb.connect(VECTORSTORE_DIR)
table = db.open_table(TABLE_NAME)

print(f"\u2705 Таблица '{TABLE_NAME}' открыта: {table.count_rows()} строк")

✅ Таблица 'chunks' открыта: 128 строк


In [4]:
# from agent import RAGAgent, RAGAgentAnswer, TOOLS_SCHEMA, get_tool_functions

# TOOL_FUNCTIONS = get_tool_functions(table, client)

# agent = RAGAgent(
#     client=client,
#     model=MODEL,
#     tools_schema=TOOLS_SCHEMA,
#     tool_functions=TOOL_FUNCTIONS,
# )

# print("\u2705 Агент готов")

In [5]:
from agent import RAGAgent, RAGAgentAnswer
from embedder import Embedder
from retriever import Retriever

retriever = Retriever(client=client, model=MODEL, top_k=1)
agent = RAGAgent(client=client, model=MODEL, retriver=retriever)

agent.run("В 2019 году как изменился индекс потребительской уверенности?", verbose=True)

Preset 'granite_vision_v4' already registered for ChartExtractionVlmEngineOptions


Найдено чанков: 1, Символов контекста: 959
Ответ: Во II квартале 2019 года индекс потребительской уверенности (ИПУ) повысился на 1 процентный пункт до значения (-15)%.


RAGAgentAnswer(dataset_row_id=None, answer='Во II квартале 2019 года индекс потребительской уверенности (ИПУ) повысился на 1 процентный пункт до значения (-15)%.', retrieved_chunks=[Chunk(text='Индекс потребительской уверенности Росстата является важнейшей компонентой сводного индекса экономического настроения (ИЭН ВШЭ), который ежеквартально рассчитывается Центром конъюнктурных исследований и интегрально характеризует состояние делового климата в экономике страны.\n\n## Основные итоги II квартала 2019 года\n- Индекс потребительской уверенности (ИПУ) повысился на 1 процентный пункт (п. п.) до значения (-15)%\n- Улучшение личного материального положения за последние 12 месяцев констатировали 11% респондентов, а его ухудшение - 32% (кварталом ранее - 10 и 34%)\n- Ухудшения личного материального положения в течение следующих 12 месяцев ожидают 19% респондентов (кварталом ранее - 23%)\n- Позитивно оценили произошедшие за год изменения в экономике России 12% участников опроса, негативно - 4

In [6]:
from lance_db import read_all, get_by_ids

df = read_all()
print(df.groupby("year").chunk_id.agg(["min", "max", "count"]))
print("дублей текста:", df.duplicated("text").sum())

for cid in (4, 13):
    print("---", cid)
    for r in get_by_ids(2019, [cid - 1, cid, cid + 1]):
        print(r["chunk_id"], repr(r["text"][:70]))

      min  max  count
year                 
2019    0   51     52
2025   52  127     76
дублей текста: 2
--- 4
3 'Материал подготовлен в ходе проведения исследования в рамках Программы'
4 '- © Национальный исследовательский университет «Высшая школа экономики'
5 'Центр конъюнктурных исследований Института статистических исследований'
--- 13
12 'Улучшились, хотя и в разной степени, все компоненты ИПУ. Частные индек'
13 '. п. Несмотря на позитивную коррекцию, и сам ИПУ, и его компоненты пок'
14 'Рис. 2. Частные индексы изменений экономической ситуации в России\n\nи л'


---
## 2. Загрузка тестового датасета

Датасет: `data/test_dataframe.xlsx` — 150 вопросов с ground truth.

| Колонка | Описание |
|---------|----------|
| `id` | Уникальный идентификатор вопроса |
| `question` | Вопрос на естественном языке |
| `ground_truth_chunks` | Эталонные чанки (строковое представление списка) |
| `document` | Год отчёта (2019 / 2025) |

In [7]:
df_test = pd.read_excel("dataset/test_dataframe.xlsx")
print(f"Размер: {df_test.shape}")
print(f"Колонки: {list(df_test.columns)}")
# df_test = df_test.head(20)
df_test.head(3)

Размер: (150, 4)
Колонки: ['id', 'question', 'relevant_text', 'document']


,id,question,relevant_text,document
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,['Как мы предполагали при анализе результатов ...,2019
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...","['Одним из главных негативных факторов, тормоз...",2025
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...","['Очевидно, что основным фактором, определяющи...",2019


---
## 3. Прогон агента на тестовом датасете

Запускаем агента на каждом вопросе и сохраняем `list[RAGAgentAnswer]`.  
Это **дорогая операция** (150 вопросов × N API-вызовов), поэтому используем многопоточность.

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def _run_single(idx: int, row_id: str, question: str, agent: RAGAgent) -> tuple[int, RAGAgentAnswer]:
    """Один вызов агента (для параллельного запуска)."""
    try:
        result = agent.run(question, verbose=False, dataset_row_id=row_id)
    except Exception as e:
        print(f"  [Ошибка] вопрос {idx}: {e}")
        result = RAGAgentAnswer(dataset_row_id=row_id, answer=f"[Ошибка: {e}]", retrieved_chunks=None)
    return idx, result


def run_evaluation(df: pd.DataFrame, agent: RAGAgent, max_workers: int = 4) -> list[RAGAgentAnswer]:
    """Прогнать агента на всех вопросах (многопоточно)."""
    results: list[RAGAgentAnswer | None] = [None] * len(df)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_run_single, idx, row["id"], row["question"], agent): idx for idx, row in df.iterrows()
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="Прогон агента"):
            idx, result = future.result()
            results[idx] = result

    return results

In [9]:
results = run_evaluation(df_test, agent)

print(f"\nВсего результатов: {len(results)}")
print(f"С чанками: {sum(1 for r in results if r.retrieved_chunks)}")
print(f"Без чанков (None): {sum(1 for r in results if not r.retrieved_chunks)}")

Прогон агента:   0%|          | 0/150 [00:00<?, ?it/s]

Прогон агента: 100%|██████████| 150/150 [06:26<00:00,  2.58s/it]


Всего результатов: 150
С чанками: 150
Без чанков (None): 0


---
## 4. Оценка через `evaluation.evaluate()`

Весь расчёт метрик — **одна строка**. Модуль `evaluation.py` принимает:
- `answers`: `list[RAGAgentAnswer]` — ответы агента
- `dataset`: `pd.DataFrame` — тестовый датасет с ground truth
- `client` + `model` — для вычисления Faithfulness (LLM-as-judge)

Возвращает `pd.DataFrame` с метриками по каждому вопросу.

In [10]:
df_test

,id,question,relevant_text,document
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,['Как мы предполагали при анализе результатов ...,2019
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...","['Одним из главных негативных факторов, тормоз...",2025
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...","['Очевидно, что основным фактором, определяющи...",2019
3,d52e6df3-d5ad-404e-9da9-f6f1855d16df,Какой процент респондентов в 2025 году затрудн...,"['затрудняюсь ответить — 0,6%']",2025
4,536bf426-d85f-472c-b434-6796078cf647,"Согласно данным 2019 года, какой был баланс от...","['В результате, баланс отрицательных и положит...",2019
...,...,...,...,...
145,17fc34cc-b8eb-49f0-b384-1d1db1998846,"Согласно данным 2025 года, какой процент респо...","['- затрудняюсь ответить — 0,3%']",2025
146,60a79f64-8f20-44dc-955e-8e8deee332e8,обобщающий индекс потребительской уверенности ...,['Обобщающий индекс потребительской уверенност...,2025
147,f5995543-80ec-4eb6-a0e2-fdf0b5d0de05,В 2025 году что является важнейшим конъюнктурн...,['Именно ожидания людей и бизнеса являются важ...,2025
148,635706a4-8be4-4550-9782-02bbaf29e2cd,Как изменилось восприятие респондентов относит...,['аспределение мнений респондентов сместилось ...,2019


In [11]:
results

[RAGAgentAnswer(dataset_row_id='e876ec3b-7442-4145-99d0-d6bb85fd982d', answer='В контексте нет данных о полном 2019 г. по сравнению с предыдущим годом. Указано лишь, что в I–II кварталах 2019 г. позитивная коррекция ИПУ продолжилась с минимальными темпами — по 1 п.п. за каждый квартал, после снижения во второй половине 2018 г. с (-8) до (-17%).', retrieved_chunks=[Chunk(text='## Позиция России в рейтинге стран ЕС\n\nВ рейтинге европейских стран, составленном на основе значений индекса потребительской уверенности, Россия традиционно занимает третье место с конца, превосходя Грецию и Болгарию.\n\nРис. 7. Индекс потребительской уверенности в России и ряде стран ЕС во II квартале 2019 г. Балансы, %\n\nИсточники: Россия - Росстат; страны ЕС - электронная база данных Евростата c учетом пересмотра состава индекса в январе 2019 г.\n\n## Выводы\nКак мы и предполагали при анализе результатов предыдущего раунда обследования, позитивная коррекция индекса потребительской уверенности, начавшаяся в I

In [12]:
from evaluation import evaluate

df_metrics = evaluate(
    answers=results,
    dataset=df_test,
    client=client,
    # model="openai/gpt-5.6-luna",
    # model = MODEL,
    model="openai/gpt-6-luna",
    overlap_threshold=0.3,
    compute_faithfulness=True,
)

print(f"\u2705 Оценка завершена: {df_metrics.shape[0]} строк, {df_metrics.shape[1]} колонок")
df_metrics.head(10)

Retrieval metrics: 100%|██████████| 150/150 [00:00<00:00, 1887.95it/s]

✅ Оценка завершена: 150 строк, 9 колонок


,id,question,precision,recall,f1,mrr,map,ndcg,faithfulness
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,1.0,1.0,1.0,1.0,1.0,1.0,0.0
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...",1.0,1.0,1.0,1.0,1.0,1.0,0.0
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...",1.0,1.0,1.0,1.0,1.0,1.0,0.0
3,d52e6df3-d5ad-404e-9da9-f6f1855d16df,Какой процент респондентов в 2025 году затрудн...,1.0,1.0,1.0,1.0,1.0,1.0,0.0
4,536bf426-d85f-472c-b434-6796078cf647,"Согласно данным 2019 года, какой был баланс от...",1.0,1.0,1.0,1.0,1.0,1.0,0.0
5,e3453b4c-2d2a-4569-899d-f731158e2d36,"Согласно данным 2025 года, какие социальные по...",1.0,1.0,1.0,1.0,1.0,1.0,0.0
6,a3ae1b1c-a296-4e85-b8df-6ca1ed2093d0,В 2025 году какова связь между потребительским...,1.0,1.0,1.0,1.0,1.0,1.0,0.0
7,25a2eff0-2893-471f-a145-20aa63633e13,В 2019 году какое место в рейтинге стран ЕС по...,1.0,1.0,1.0,1.0,1.0,1.0,0.0
8,49287b9e-2dad-4902-a389-bc0a34494dfc,Какая доля населения в 2025 году в основном фо...,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,6192474e-db88-47a1-9c27-5c5e89d5e430,Какой процент респондентов оказался в затрудни...,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## 5. Анализ результатов

In [13]:
metric_cols = ["precision", "recall", "f1", "mrr", "map", "ndcg", "faithfulness"]
display_names = ["Precision@K", "Recall@K", "F1@K", "MRR", "MAP", "NDCG@K", "Faithfulness"]

summary = df_metrics[metric_cols].mean()
summary.index = display_names

print("=" * 45)
print("  ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА RAG-АГЕНТА")
print("=" * 45)
for name, val in summary.items():
    bar = "\u2588" * int(val * 30) + "\u2591" * (30 - int(val * 30))
    print(f"  {name:<15} {val:.4f}  {bar}")
print("=" * 45)

  ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА RAG-АГЕНТА
  Precision@K     0.8800  ██████████████████████████░░░░
  Recall@K        0.8478  █████████████████████████░░░░░
  F1@K            0.8567  █████████████████████████░░░░░
  MRR             0.8800  ██████████████████████████░░░░
  MAP             0.8478  █████████████████████████░░░░░
  NDCG@K          0.8546  █████████████████████████░░░░░
  Faithfulness    0.0000  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░


In [14]:
# Сводная таблица
df_summary = pd.DataFrame(
    list(zip(display_names, summary.values)),
    columns=["Метрика", "Значение"],
)
df_summary["Значение"] = df_summary["Значение"].round(4)
df_summary

,Метрика,Значение
0,Precision@K,0.8800
1,Recall@K,0.8478
2,F1@K,0.8567
3,MRR,0.8800
4,MAP,0.8478
5,NDCG@K,0.8546
6,Faithfulness,0.0000


In [15]:
# Распределение метрик
df_metrics[metric_cols].describe().round(4)

,precision,recall,f1,mrr,map,ndcg,faithfulness
count,150.0000,150.0000,150.0000,150.0000,150.0000,150.0000,150.0
mean,0.8800,0.8478,0.8567,0.8800,0.8478,0.8546,0.0
std,0.3261,0.3427,0.3328,0.3261,0.3427,0.3346,0.0
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
25%,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0
50%,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0
75%,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0
max,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0


In [16]:
# Худшие вопросы по F1
worst = df_metrics.nsmallest(5, "f1")[["id", "question", "precision", "recall", "f1", "faithfulness"]]
worst

,id,question,precision,recall,f1,faithfulness
8,49287b9e-2dad-4902-a389-bc0a34494dfc,Какая доля населения в 2025 году в основном фо...,0.0,0.0,0.0,0.0
9,6192474e-db88-47a1-9c27-5c5e89d5e430,Какой процент респондентов оказался в затрудни...,0.0,0.0,0.0,0.0
18,4a0cb62e-599d-447e-be34-13df205b7296,В 2025 году какая часть общества преимуществен...,0.0,0.0,0.0,0.0
36,6042cae5-d48a-41fb-abf7-e4855623659f,"Интересно, какой процент респондентов в 2019 г...",0.0,0.0,0.0,0.0
44,6c80f87e-e182-45fa-a989-8c6cfb067322,"Согласно данным 2025 года, какой процент респо...",0.0,0.0,0.0,0.0


In [ ]:
import numpy as np
print(np.mean([len(r.retrieved_chunks) for r in results]))   
print(max(sum(len(c.text) for c in r.retrieved_chunks) for r in results))  

1.0
1427


In [18]:
import ast
from evaluation import _token_overlap_recall
from lance_db import search

for _, row in df_test[df_test["id"].isin(ids)].iterrows():
    gts = ast.literal_eval(row["relevant_text"])
    year = int(row["document"])
    hits = search(retriever.embedder.embed_query(row["question"]), 10, year=year)
    place = next((i + 1 for i, h in enumerate(hits)
                  if any(_token_overlap_recall(retriever.expand(h), g) >= 0.3 for g in gts)), None)
    print(f"место эталона в top-10: {place}\t{row['question'][:60]}")

NameError: name 'ids' is not defined

In [19]:
# Лучшие вопросы по F1
best = df_metrics.nlargest(5, "f1")[["id", "question", "precision", "recall", "f1", "faithfulness"]]
best

,id,question,precision,recall,f1,faithfulness
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,1.0,1.0,1.0,0.0
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...",1.0,1.0,1.0,0.0
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...",1.0,1.0,1.0,0.0
3,d52e6df3-d5ad-404e-9da9-f6f1855d16df,Какой процент респондентов в 2025 году затрудн...,1.0,1.0,1.0,0.0
4,536bf426-d85f-472c-b434-6796078cf647,"Согласно данным 2019 года, какой был баланс от...",1.0,1.0,1.0,0.0
